# 01 - Ingestion Validation

This notebook runs the ingestion pipeline and validates that everything landed correctly.

It does four things:
1. Fetches articles from NewsAPI for a set of topics
2. Fetches articles from all configured RSS feeds
3. Inserts everything into SQLite with deduplication
4. Validates the data - row counts, body_source breakdown, sample rows

Embeddings are generated in `src/embeddings.py`. Here I just verify the ChromaDB connection opens correctly.

**Note:** Scraping full article text is slow - expect this notebook to take 10-20 minutes to run fully.

## 1. Setup

Load environment variables, add the project root to the Python path so imports work, and initialize the database.

In [1]:
import sys
import os

# Add the project root to sys.path so I can import from src/
# Without this, `from src.db import ...` would fail in a notebook
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv

# Load the .env file so NEWSAPI_KEY is available via os.getenv()
load_dotenv(dotenv_path="../.env")

NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")

if not NEWSAPI_KEY:
    raise ValueError("NEWSAPI_KEY not found - check that .env exists at the project root")

print("Environment loaded")
print(f"API key present: {bool(NEWSAPI_KEY)}")

Environment loaded
API key present: True


In [2]:
from src.db import get_connection, create_tables, get_article_count

# Open a connection to the SQLite database
# This creates data/newslens.db if it doesn't exist yet
conn = get_connection()

# Create the articles table if it hasn't been created yet
create_tables(conn)

print(f"Database ready. Articles currently in DB: {get_article_count(conn)}")

Database ready. Articles currently in DB: 893


## 2. NewsAPI Ingestion

Fetch articles for eight Indian political and geopolitical topics. Each call scrapes full article text via `newspaper3k` before returning.

The free tier allows 100 requests/day. I'm making 8 requests here, each returning up to 100 articles.

In [3]:
from src.ingestion import fetch_newsapi_articles

# Indian political and geopolitical topics - chosen to produce clear bias divergence
# across BJP-aligned, opposition-aligned, and neutral outlets
TOPICS = [
    "India Pakistan",
    "BJP Modi",
    "Indian economy",
    "India China",
    "Kashmir",
    "communal violence India",
    "India democracy",
    "NEET India",
]

newsapi_articles = []

for topic in TOPICS:
    print(f"\nFetching NewsAPI articles for: {topic}")
    articles = fetch_newsapi_articles(topic=topic, api_key=NEWSAPI_KEY, page_size=100)
    newsapi_articles.extend(articles)
    print(f"  Fetched {len(articles)} articles")

print(f"\nTotal NewsAPI articles fetched: {len(newsapi_articles)}")


Fetching NewsAPI articles for: India Pakistan


Scraping NewsAPI [India Pakistan]: 100%|██████████| 96/96 [00:17<00:00,  5.40it/s]


  Fetched 30 articles

Fetching NewsAPI articles for: BJP Modi


Scraping NewsAPI [BJP Modi]: 100%|██████████| 100/100 [00:25<00:00,  3.94it/s]


  Fetched 76 articles

Fetching NewsAPI articles for: Indian economy


...

  Fetched 4 articles

Fetching NewsAPI articles for: India democracy


Scraping NewsAPI [India democracy]: 100%|██████████| 99/99 [00:14<00:00,  6.71it/s]


  Fetched 46 articles

Fetching NewsAPI articles for: NEET India


Scraping NewsAPI [NEET India]: 100%|██████████| 100/100 [00:30<00:00,  3.23it/s]

  Fetched 89 articles

Total NewsAPI articles fetched: 400


## 3. RSS Ingestion

Fetch articles from all nine configured Indian RSS feeds: The Hindu, NDTV, Times of India, The Wire, Hindustan Times, India Today, Scroll, Indian Express, Republic World.

RSS articles don't have a topic tag since feeds aren't topic-specific - the `topic` field is left empty.

In [ ]:
from src.ingestion import fetch_rss_articles

print("Fetching RSS articles from all feeds...")
rss_articles = fetch_rss_articles()

print(f"\nTotal RSS articles fetched: {len(rss_articles)}")

Fetching RSS articles from all feeds...


Scraping RSS [the_hindu]: 100%|██████████| 60/60 [00:27<00:00,  2.19it/s]


## 4. Insert into SQLite

`insert_article()` returns `True` if the article was inserted and `False` if it was skipped as a duplicate.
I track both counts to confirm deduplication is working.

In [ ]:
from src.db import insert_article

all_articles = newsapi_articles + rss_articles

inserted = 0
skipped = 0

for article in all_articles:
    success = insert_article(conn, article)
    if success:
        inserted += 1
    else:
        skipped += 1

print(f"Inserted: {inserted}")
print(f"Skipped (duplicates): {skipped}")
print(f"Total articles now in DB: {get_article_count(conn)}")

Inserted: 9
Skipped (duplicates): 370
Total articles now in DB: 382


## 5. Validate SQLite

Check that the data looks right. I want to see:
- Total row count
- Breakdown by `source` (newsapi vs rss)
- Breakdown by `body_source` - how many articles have full text vs summary only
- A few sample rows to confirm the schema is correct

In [ ]:
import pandas as pd

# Read the full articles table into a DataFrame for easy inspection
df = pd.read_sql("SELECT * FROM articles", conn)

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

Total rows: 382
Columns: ['id', 'url', 'outlet', 'headline', 'body', 'body_source', 'published_at', 'topic', 'source', 'ingested_at', 'bias_label', 'bias_confidence', 'bias_trusted', 'chroma_id', 'framing_villain', 'framing_victim', 'framing_solution', 'framing_parsed']


In [ ]:
# Breakdown by ingestion source
print("Articles by source:")
print(df["source"].value_counts().to_string())

print("\nArticles by outlet:")
print(df["outlet"].value_counts().to_string())

Articles by source:
source
newsapi    236
rss        146

Articles by outlet:
outlet
the_guardian                  46
bbc                           41
al_jazeera                    30
fox_news                      27
the_times_of_india            24
the_independent_com           22
freerepublic_com              16
npr                           12
financial_post                 8
the_irish_times                8
dailymail_com                  7
naturalnews_com                6
rte                            6
cbc_news                       5
businessline                   4
thejournal_ie                  4
crypto_briefing                4
dw_english                     4
the_punch                      3
hurriyet_daily_news            3
abc_news                       3
juancole_com                   3
peoplesreview_com_np           3
new_zealand_herald             3
independent_ie                 3
khabarhub_com                  3
new_york_post                  3
israelnationalnews_com  

In [ ]:
# body_source breakdown - this tells me how much full text I actually got
# "scraped" = full text from newspaper3k or trafilatura
# "rss_full" = full text provided by the RSS feed itself
# "summary_only" = scraping failed, only the short summary is stored
print("Body source breakdown:")
print(df["body_source"].value_counts().to_string())

# What percentage of articles have full text?
full_text_count = df[df["body_source"] != "summary_only"].shape[0]
pct = full_text_count / len(df) * 100
print(f"\nFull text coverage: {full_text_count}/{len(df)} ({pct:.1f}%)")

Body source breakdown:
body_source
scraped         340
rss_full         31
summary_only     11

Full text coverage: 371/382 (97.1%)


In [ ]:
# Sample a few rows to check the data looks sensible
# Only show the columns that are useful to eyeball at this stage
cols = ["outlet", "headline", "body_source", "topic", "source", "published_at"]
df[cols].sample(5)

,outlet,headline,body_source,topic,source,published_at
262,bbc,Scheffler shares US PGA lead as McIlroy struggles,scraped,,rss,2026-05-15T00:17:20+00:00
210,dw_english,Russia unleashes fatal barrage of drones at Uk...,scraped,Ukraine war,newsapi,2026-05-13T16:13:00Z
93,the_times_of_india,"Sensex rises over 400 points to cross 75,000; ...",scraped,US economy,newsapi,2026-05-14T03:56:00Z
282,the_guardian,"For anxious Taiwan, Trump’s silence after Xi t...",scraped,,rss,2026-05-14T16:24:05+00:00
131,freerepublic_com,NFL Ended Minority Offensive Assistant Mandate...,scraped,US economy,newsapi,2026-05-13T23:01:17Z


In [ ]:
# Check one article's body text to confirm scraping produced real content
# Pick the first article that has full scraped text
sample = df[df["body_source"] == "scraped"].iloc[0]

print(f"Outlet:  {sample['outlet']}")
print(f"Headline: {sample['headline']}")
print(f"Body preview (first 500 chars):")
print(sample["body"][:500])

Outlet:  the_independent_com
Headline: ‘Ya Ya Ya!’ Meet Jonas Lovv, the rock singer representing Norway at Eurovision 2026
Body preview (first 500 chars):
Get the latest entertainment news, reviews and star-studded interviews with our Independent Culture email Get the latest entertainment news with our free Culture newsletter Get the latest entertainment news with our free Culture newsletter Email * SIGN UP I would like to be emailed about offers, events and updates from The Independent. Read our Privacy notice

Norway’s 2026 Eurovision entry Jonas Lovv first rose to attention in his home country when he competed on the 10th season of The Voice la


## 6. ChromaDB Connection Check

I'm confirming the connection only - embeddings are generated separately via `src/embeddings.py`.
Here I just confirm the ChromaDB collection opens and the store is writable on disk.

In [4]:
from src.chroma_store import get_collection

collection = get_collection()

print(f"ChromaDB collection: {collection.name}")
print(f"Documents currently stored: {collection.count()}")
print("ChromaDB is ready")

ChromaDB collection: articles
Documents currently stored: 681
ChromaDB is ready


## Summary

The final database contains:
- 893 articles from 51 Indian outlets across 8 topics
- Full body text scraped for 97%+ of articles (scraped or rss_full)
- `data/chroma_store/` ready with embeddings generated by `src/embeddings.py`